# E5c: HCC-TACE-Seg external test with the axis fix

**Revision plan, 00 log section 18 (approved 17 Oct 2026).** While building E6 we found that the submitted external
pipeline (Week 10g, reproduced exactly by E5) gave the CNN every HCC-TACE-Seg volume in SimpleITK axis order (k, j, i),
while the CNN was trained on (i, j, k) arrays read with nibabel (00 log 17.9). Radiomics does not depend on axis
order. E5c measures what the fix changes.

**Version 2 (17 Oct 2026):** orientation from each image header (amendment 18.1); version 1 stopped at the orientation
check before any model ran.

**Two reproduction gates first:** the historical settings must reproduce the submitted external CNN predictions
(0/103 and 5/103), and the E5 settings must reproduce the saved E5 v2 predictions. Otherwise the notebook stops.

**Variants:** `historical_W150C50` (submitted), `E5_corrected` (E5 v2), `axis_fixed_iso1mm` (E5 with only the axis
order fixed and the orientation matched to the internal data, taken from each image header — amendment 18.1), `axis_fixed_E6rules` (**primary corrected result**:
the E6 primary rules — in-plane 0.78 mm, slices kept, orientation matched, 0–255 crops as internally).

**Reported:** HCC sensitivity with exact 95% CI and predicted-class counts for every model and variant; primary subset
clean 89, all 103 also shown. HCC-only cohort: no AUC or specificity (E6 gives those).

**Runtime:** GPU runtime (L4 is enough), about 1.5 hours. **Safety:** inputs are only read; outputs go to a new
folder `liver_project/revision_2026-10/E5c_<time>/`.

In [ ]:
# Cell 1 — Setup: installs, paths, input checks, output folder (E5's cell; output folder E5c_<time>)
import os, sys, json, hashlib, platform, datetime, re, time, warnings, subprocess, shutil, tarfile
from pathlib import Path
warnings.filterwarnings('ignore')

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount(os.environ['COLAB_DRIVE_MOUNT'])  # see docs/revision_notebooks.md
    ROOT = Path(os.environ['LIVER_PROJECT_ROOT'])
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'SimpleITK', 'monai', 'lightgbm', 'openpyxl', 'nibabel'], check=True)
    radiomics_ok = False
    for spec in (['pyradiomics'], ['git+https://github.com/AIM-Harvard/pyradiomics.git']):
        if subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + spec).returncode == 0:
            radiomics_ok = True
            break
    if not radiomics_ok:
        raise RuntimeError('PyRadiomics could not be installed. Send this output for review.')
else:
    ROOT = Path(os.environ['E5_TEST_ROOT'])

import numpy as np
import pandas as pd
import torch
import SimpleITK as sitk
import nibabel as nib
from concurrent.futures import ThreadPoolExecutor
import joblib
from radiomics import featureextractor
from monai.networks.nets import resnet18
from scipy.ndimage import binary_dilation
from scipy.stats import beta

PROJECT_DIR = ROOT / 'liver_project'
DATA_DIR = ROOT / 'liver_data'
REVISION_DIR = PROJECT_DIR / 'revision_2026-10'
E2_DIR = REVISION_DIR / 'E1_E2_20261017T032941Z'
E3_DIR = REVISION_DIR / 'E3_20261017T041700Z'
EXT_DIR = PROJECT_DIR / 'external_validation' / 'HCC-TACE-Seg'
OLD_PREFIX = os.environ['LIVER_HISTORICAL_PREFIX']   # paths stored in the Week-10 manifest (folder moved 7 Sep 2026)
SEED = 42
CLASSES = ['HCC', 'ICC', 'CHCC']
PHASE_ORDER = ['P', 'C1', 'C2', 'C3']

INPUTS = {
    'manifest_v4': (EXT_DIR / 'nifti_v4_liver_adapter' / 'manifests' / 'external_manifest_for_week10_tumor_only_FILTERED_liver_adapter.csv',
                    'e428d3ad7b5928622e51b78c1c3346b9b168ef9e6b6205b00e33a61f9f84e5ad'),
    'dicom_catalog': (EXT_DIR / 'nifti' / 'logs' / 'dicom_catalog.csv', '515453cc76f824b0803a4d885e0065e395198e4ba3521c2df79b21762e395e62'),
    'historical_external': (PROJECT_DIR / 'results' / 'external_HCC_TACE_SEG_liver_adapter_fixed_103' / 'external_predictions_with_phase_audit.csv',
                            'a5e412a63778013130c90bb50ae37591e16e4d85964053d15abf83b1443424fb'),
    'patient_data': (DATA_DIR / 'patient_data.csv', '8cd4eb9405a3987c35d7ba4e92893c474f20b6a2791ab23bac2c47ef5532b185'),
    'split': (PROJECT_DIR / 'splits' / 'train_val_test.json', 'cfbe3cb6f9b1901c3d672042e7918a95051d841d328d8353492f91376617513e'),
    'radiomics_features': (PROJECT_DIR / 'radiomics' / 'radiomics_features.parquet', 'caac3d2cf4f273bd6eebd48154a1dc4bf20adbbb7a896b3c4947a910fd01e156'),
    'e2_oof_meta': (E2_DIR / 'E2_fusion_oof_meta_features_noshape.csv', 'c20d23d3d0a7df871212f6f937bc2024e686c11dc6975350329cbe186f058d12'),
    'e2_noshape_test': (E2_DIR / 'E2_radiomics_noshape_test_predictions.csv', '9aa460d2ebe597b8b116006af9895fbf9f6fe55f87a9268bef69d69f2830db90'),
    'e3_d_predictions': (E3_DIR / 'E3B_D_phase_dropout_predictions.csv', '4c1aed86c7e8095390b8c4313cb69e12c31bd87a6e67a518e23e3e3e95470ee6'),
    'e5_v2_predictions': (REVISION_DIR / 'E5_20261017T062958Z' / 'E5_predictions.csv', '5100abe40d584c69872099121175d71d13a67ab407e684f8ae497391e12c1149'),
}
for f in range(5):
    INPUTS[f'fold_{f}'] = (PROJECT_DIR / 'splits' / f'fold_{f}.json', None)
MODEL_FILES = {
    **{f'w3_{f}': PROJECT_DIR / 'checkpoints_w3' / f'fold_{f}_best.pt' for f in range(5)},
    **{f'w4_{f}': PROJECT_DIR / 'checkpoints_w4' / f'fold_{f}_best.pt' for f in range(5)},
    **{f'd_{f}': E3_DIR / 'D_phase_dropout' / f'fold_{f}.pt' for f in range(5)},
    **{f'rad_{f}': E2_DIR / 'e2_radiomics_noshape_fold_models' / f'fold_{f}.joblib' for f in range(5)},
}

def sha256_of(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for block in iter(lambda: handle.read(1 << 20), b''):
            digest.update(block)
    return digest.hexdigest()

problems, input_hashes = [], {}
for key, (path, expected) in INPUTS.items():
    if not path.exists():
        problems.append(f'missing: {path}')
        continue
    input_hashes[key] = sha256_of(path)
    if expected and input_hashes[key] != expected:
        (problems if IN_COLAB else []).append(f'changed: {path.name}')
problems += [f'missing: {p}' for p in MODEL_FILES.values() if not p.exists()]
if problems:
    raise RuntimeError('STOP — input check failed:\n' + '\n'.join(problems))

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
import sklearn, scipy, lightgbm, radiomics, monai
VERSIONS = {'python': platform.python_version(), 'torch': torch.__version__, 'monai': monai.__version__,
            'SimpleITK': sitk.Version_VersionString() if hasattr(sitk, 'Version_VersionString') else 'unknown',
            'pyradiomics': getattr(radiomics, '__version__', 'unknown'), 'lightgbm': lightgbm.__version__,
            'scikit-learn': sklearn.__version__, 'numpy': np.__version__, 'device': str(DEVICE)}
RUN_ID = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
OUT_DIR = REVISION_DIR / f'E5c_{RUN_ID}'
N_IO = 4 if IN_COLAB else 1
N_JOBS = os.cpu_count() if IN_COLAB else 1
OUT_DIR.mkdir(parents=False, exist_ok=False)
(OUT_DIR / 'inputs').mkdir()
print('Inputs and models checked:', len(INPUTS) + len(MODEL_FILES))
print('Versions:', VERSIONS)
print('Output folder:', OUT_DIR.name)

In [ ]:
# Cell 2 — Download the official TCIA clinical file and extract age and sex (stops if the format is unclear)
CLINICAL_URL = 'https://www.cancerimagingarchive.net/wp-content/uploads/HCC-TACE-Seg_clinical_data-V2.xlsx'
if IN_COLAB:
    import urllib.request
    clinical_path = OUT_DIR / 'inputs' / 'HCC-TACE-Seg_clinical_data-V2.xlsx'
    request = urllib.request.Request(CLINICAL_URL, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(request, timeout=120) as response:
        clinical_path.write_bytes(response.read())
    sheets = pd.read_excel(clinical_path, sheet_name=None)
else:
    clinical_path = OUT_DIR / 'inputs' / 'clinical_test.csv'
    shutil.copy2(os.environ['E5_TEST_CLINICAL'], clinical_path)
    sheets = {'test': pd.read_csv(clinical_path)}
    if os.environ.get('E5_TEST_GLOSSARY'):
        sheets['glossary'] = pd.read_csv(os.environ['E5_TEST_GLOSSARY'])
input_hashes['tcia_clinical'] = sha256_of(clinical_path)

def normalized(name):
    return re.sub(r'[^a-z0-9]+', ' ', str(name).lower()).strip()

def find_clinical_table(sheets):
    for sheet_name, table in sheets.items():
        for column in table.columns:
            values = table[column].dropna().astype(str).str.strip()
            if len(values) and values.str.fullmatch(r'HCC_\d+').mean() > 0.9:
                return sheet_name, table, column
    return None, None, None

sheet_name, table, id_col = find_clinical_table(sheets)
age_cols = [c for c in (table.columns if table is not None else []) if re.search(r'\bage\b', normalized(c))]
sex_cols = [c for c in (table.columns if table is not None else []) if re.search(r'\b(sex|gender)\b', normalized(c))]
print('Sheets:', list(sheets))
if table is not None:
    print('Using sheet:', sheet_name, '| ID column:', id_col, '| age columns:', age_cols, '| sex columns:', sex_cols)
if table is None or len(age_cols) != 1 or len(sex_cols) != 1:
    for name, t in sheets.items():
        print(f'--- sheet {name}: columns', list(t.columns))
    raise RuntimeError('STOP — could not identify a unique patient-ID, age and sex column. '
                       'Send this output; the column choice will be set explicitly.')

SEX_MAP = {'m': 1.0, 'male': 1.0, 'f': 0.0, 'female': 0.0}     # internal coding: male = 1, female = 0

def sex_tokens(series):
    return series.astype(str).str.strip().str.lower().str.replace(r'^(\d+)\.0$', r'\1', regex=True)

def sex_codes_from_glossary(sheets, sex_col):
    """Numeric codes defined in the file's own glossary sheet (TCIA V2: 'Sex | 1=Male, 2=Female')."""
    for name, t in sheets.items():
        if name == sheet_name:
            continue
        for row in [list(t.columns)] + t.astype(str).values.tolist():
            cells = [str(v).strip() for v in row]
            if cells and normalized(cells[0]) == normalized(sex_col):
                text = ' '.join(cells[1:])
                pairs = re.findall(r'(\d+)\s*=\s*(male|female)\b', text, flags=re.I)
                if pairs:
                    return {code: SEX_MAP[word.lower()] for code, word in pairs}, f'sheet {name!r}: {cells[0]} | {text}'
    return {}, None

raw_sex = sex_tokens(table[sex_cols[0]].dropna())
print('Sex values in the file:', raw_sex.value_counts().to_dict())
SEX_CODING = 'text (male/female)'
if set(raw_sex) - set(SEX_MAP):
    glossary_codes, glossary_line = sex_codes_from_glossary(sheets, sex_cols[0])
    print('Sex coding stated in the file glossary:', glossary_line)
    if sorted(glossary_codes.values()) != [0.0, 1.0]:
        raise RuntimeError('STOP — numeric sex codes and no clear male/female definition in the glossary. Send this output.')
    SEX_MAP = {**SEX_MAP, **glossary_codes}
    SEX_CODING = f'numeric, from the glossary ({glossary_line}); mapped to male = 1, female = 0'
unmapped = sorted(set(raw_sex) - set(SEX_MAP))
if unmapped:
    raise RuntimeError(f'STOP — unrecognised sex codes {unmapped}. Send this output; the coding will be set explicitly.')
mapped = raw_sex.map(SEX_MAP)
print(f'Sex after mapping to the internal coding: male {int((mapped == 1).sum())}, female {int((mapped == 0).sum())}')
raw_age = pd.to_numeric(table[age_cols[0]], errors='coerce')
print(f'Age: {raw_age.notna().sum()} numeric values, range {raw_age.min()}–{raw_age.max()}')
if raw_age.notna().sum() and not (0 < raw_age.min() and raw_age.max() < 110):
    raise RuntimeError('STOP — age values outside 0–110. Send this output.')
clinical = pd.DataFrame({
    'patient_id': table[id_col].astype(str).str.strip().str.replace(r'HCC_(\d+)', lambda m: f'HCC_{int(m.group(1)):03d}', regex=True),
    'age': pd.to_numeric(table[age_cols[0]], errors='coerce'),
    'sex': sex_tokens(table[sex_cols[0]]).map(SEX_MAP),
}).dropna(subset=['patient_id'])
conflicts = clinical.groupby('patient_id').agg(age_n=('age', 'nunique'), sex_n=('sex', 'nunique'))
conflicting_ids = set(conflicts.index[(conflicts['age_n'] > 1) | (conflicts['sex_n'] > 1)])
clinical = clinical.groupby('patient_id', as_index=False).first()
clinical.loc[clinical['patient_id'].isin(conflicting_ids), ['age', 'sex']] = np.nan
print(f'Clinical rows: {len(clinical)} patients; conflicting duplicate rows set to missing: {len(conflicting_ids)}')

In [ ]:
# Cell 3 — External manifest (paths remapped to the new folder) and mixed-scan detection
manifest = pd.read_csv(INPUTS['manifest_v4'][0])
path_cols = [c for c in manifest.columns if re.match(r'(ct|mask|liver_mask)_(P|C1|C2|C3)$', c)]
def remap(value):
    if isinstance(value, str) and value.startswith(OLD_PREFIX):
        return str(PROJECT_DIR) + value[len(OLD_PREFIX):]
    return value
for column in path_cols:
    manifest[column] = manifest[column].map(remap)
referenced = {v for c in path_cols for v in manifest[c].dropna() if isinstance(v, str) and v.strip()}
missing_files = [v for v in referenced if not Path(v).exists()]
assert len(manifest) == 103, len(manifest)
if missing_files:
    raise RuntimeError(f'STOP — {len(missing_files)} manifest files not found, e.g. {missing_files[:3]}')

catalog = pd.read_csv(INPUTS['dicom_catalog'][0], usecols=['SeriesInstanceUID', 'Modality', 'ImagePositionPatient'])
catalog = catalog[catalog['Modality'] == 'CT'].copy()
catalog['z'] = catalog['ImagePositionPatient'].map(
    lambda s: round(float(re.findall(r'[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?', s)[2]), 1) if isinstance(s, str) else np.nan)
per_series = catalog.groupby('SeriesInstanceUID').agg(n=('z', 'size'), unique_z=('z', 'nunique'))
per_series['repeats'] = per_series['n'] / per_series['unique_z']

def series_uid(path):
    match = re.search(r'__([0-9.]+)__CT', str(path))
    return match.group(1) if match else None

mixed_rows = []
for _, row in manifest.iterrows():
    for phase in PHASE_ORDER:
        uid = series_uid(row.get(f'ct_{phase}'))
        if uid and uid in per_series.index and per_series.loc[uid, 'repeats'] >= 1.9:
            mixed_rows.append({'patient_id': row['patient_id'], 'channel': phase, 'series_uid': uid,
                               'files': int(per_series.loc[uid, 'n']), 'unique_positions': int(per_series.loc[uid, 'unique_z']),
                               'repeats': float(per_series.loc[uid, 'repeats'])})
mixed = pd.DataFrame(mixed_rows)
MIXED_IDS = set(mixed['patient_id']) if len(mixed) else set()
ALL_IDS = manifest['patient_id'].astype(str).tolist()
CLEAN_IDS = [p for p in ALL_IDS if p not in MIXED_IDS]
mixed.to_csv(OUT_DIR / 'E5_mixed_scan_inputs.csv', index=False)
print(f'Mixed-scan inputs: {len(mixed)} in {len(MIXED_IDS)} patients -> clean cohort {len(CLEAN_IDS)} patients')
if IN_COLAB and (len(mixed), len(MIXED_IDS)) != (16, 14):
    raise RuntimeError('STOP — mixed-scan count differs from the 17 Oct audit (16 inputs, 14 patients). Send this output.')

clinical_ext = pd.DataFrame({'patient_id': ALL_IDS}).merge(clinical, on='patient_id', how='left')
print(f'Observed age for {clinical_ext["age"].notna().sum()}/103, observed sex for {clinical_ext["sex"].notna().sum()}/103 '
      f'(male {int((clinical_ext["sex"] == 1).sum())}, female {int((clinical_ext["sex"] == 0).sum())})')
clinical_ext.to_csv(OUT_DIR / 'E5_external_clinical_joined.csv', index=False)

In [ ]:
# Cell 4 — Preprocessing: E5/Week-10g code for the gates, plus the axis-fixed variants (00 log 18)
TARGET_SPACING_MM = (1.0, 1.0, 1.0)
CROP_SIZE = (96, 96, 96)
TUMOR_DILATION_MM = 15
INPLANE_MM = 400.0 / 512                      # internal physical pixel size (E6 primary rule)

def resample_to_isotropic(arr, original_spacing, target_spacing=TARGET_SPACING_MM, is_mask=False):   # Week 10g
    img = sitk.GetImageFromArray(arr)
    img.SetSpacing(tuple(original_spacing))
    new_size = [int(round(sz * os_ / ts_)) for sz, os_, ts_ in zip(img.GetSize(), img.GetSpacing(), target_spacing)]
    resampler = sitk.ResampleImageFilter()
    resampler.SetOutputSpacing(target_spacing)
    resampler.SetSize(new_size)
    resampler.SetInterpolator(sitk.sitkNearestNeighbor if is_mask else sitk.sitkLinear)
    resampler.SetDefaultPixelValue(0 if is_mask else -1024)
    out = sitk.GetArrayFromImage(resampler.Execute(img))
    return out.astype(np.uint8) if is_mask else out.astype(np.float32)

def window_historical(hu):          # Week 10g: W150/C50 -> -25..125 HU -> 0..255
    lo, hi = 50 - 150 / 2, 50 + 150 / 2
    return ((np.clip(hu, lo, hi) - lo) / (hi - lo) * 255).astype(np.float32)

def window_corrected(hu):           # PLC-CECT release convention: -200..200 HU -> 0..255
    lo, hi = -200.0, 200.0
    return ((np.clip(hu, lo, hi) - lo) / (hi - lo) * 255).astype(np.float32)

WINDOWS = {'historical_W150C50': window_historical, 'corrected_-200_200': window_corrected}

def crop_96_around_tumor(vol, tumor_union, size=CROP_SIZE):   # Week 10g
    if tumor_union.sum() == 0:
        cz, cy, cx = vol.shape[0] // 2, vol.shape[1] // 2, vol.shape[2] // 2
    else:
        cz, cy, cx = np.argwhere(tumor_union > 0).mean(axis=0).astype(int)
    dz, dy, dx = [s // 2 for s in size]
    out = np.zeros(size, dtype=vol.dtype)
    z0, z1 = max(0, cz - dz), min(vol.shape[0], cz + dz)
    y0, y1 = max(0, cy - dy), min(vol.shape[1], cy + dy)
    x0, x1 = max(0, cx - dx), min(vol.shape[2], cx + dx)
    oz0 = dz - (cz - z0); oy0 = dy - (cy - y0); ox0 = dx - (cx - x0)
    out[oz0:oz0 + (z1 - z0), oy0:oy0 + (y1 - y0), ox0:ox0 + (x1 - x0)] = vol[z0:z1, y0:y1, x0:x1]
    return out, (cz, cy, cx)

def zscore_nonzero(x):                                          # Week 10g / Week 4
    mask = x > 0
    if mask.sum() < 100:
        return x.astype(np.float32)
    vals = x[mask]
    out = np.zeros_like(x, dtype=np.float32)
    out[mask] = (vals - vals.mean()) / (vals.std() + 1e-6)
    return out

def read_nifti(path):
    if not isinstance(path, str) or not path.strip() or path.strip().lower() in {'nan', 'none', 'null'}:
        return None, None
    img = sitk.ReadImage(path)
    return sitk.GetArrayFromImage(img).astype(np.float32), img.GetSpacing()

def compute_center_of_mass(mask):             # Week 2, unchanged
    coords = np.array(np.where(mask > 0))
    if coords.shape[1] == 0:
        return None
    return tuple(int(round(c)) for c in coords.mean(axis=1))

def crop_cube(volume, center, size):          # Week 2, unchanged
    half = size // 2
    shape = volume.shape
    starts = [c - half for c in center]
    ends = [c + size - half for c in center]
    pad_before = [max(0, -s) for s in starts]
    pad_after = [max(0, e - shape[i]) for i, e in enumerate(ends)]
    s0, s1, s2 = (max(0, s) for s in starts)
    e0, e1, e2 = (min(shape[i], e) for i, e in enumerate(ends))
    cropped = volume[s0:e0, s1:e1, s2:e2]
    if any(pad_before) or any(pad_after):
        cropped = np.pad(cropped, [(pad_before[i], pad_after[i]) for i in range(3)], mode='constant', constant_values=0)
    return cropped

# ---- orientation (E6 rule, 00 log 17.9): internal reference from the internal non-contrast scans ----
def load_nii(path):                          # Week 2, unchanged
    arr = nib.load(str(path)).get_fdata().astype(np.float32)
    while arr.ndim > 3:
        if arr.shape[-1] == 1:
            arr = arr.squeeze(-1)
        elif arr.shape[-1] == 3:
            arr = arr.mean(axis=-1)
        else:
            arr = arr[..., 0]
    return arr

BONE_0255 = 254

def orientation_descriptor(ct0255_ijk, liver_ijk):
    li, lj, lk = np.nonzero(liver_ijk)
    if li.size < 500:
        return None
    shape = liver_ijk.shape
    bone = (ct0255_ijk >= BONE_0255) & ~liver_ijk
    bi, bj, _ = np.nonzero(bone)
    area = liver_ijk.sum(axis=(0, 1)).astype(np.float64)
    k0, k1 = int(lk.min()), int(lk.max())
    centre_k = float((np.arange(len(area)) * area).sum() / area.sum())
    return {'liver_i': float(li.mean() / shape[0]), 'liver_j': float(lj.mean() / shape[1]),
            'bone_i': float(bi.mean() / shape[0]) if bi.size >= 200 else np.nan,
            'bone_j': float(bj.mean() / shape[1]) if bj.size >= 200 else np.nan,
            'bulk_k': (centre_k - k0) / max(1, k1 - k0)}

def decide(d):
    off_i, off_j = abs(d['bone_i'] - .5), abs(d['bone_j'] - .5)
    out = {'transpose': off_i > off_j, 'conf_transpose': bool(np.isfinite(off_i) and np.isfinite(off_j)
                                                              and max(off_i, off_j) >= .05 and abs(off_i - off_j) >= .03)}
    li, bj = (d['liver_j'], d['bone_i']) if out['transpose'] else (d['liver_i'], d['bone_j'])
    out['flip_i'], out['conf_flip_i'] = bool(li > .5), abs(li - .5) >= .05
    out['flip_j'] = bool(bj < .5) if np.isfinite(bj) else False
    out['conf_flip_j'] = bool(np.isfinite(bj) and abs(bj - .5) >= .05)
    out['flip_k'], out['conf_flip_k'] = bool(d['bulk_k'] > .5), abs(d['bulk_k'] - .5) >= .03
    return out

def apply_orientation(a, o):
    if o['transpose']:
        a = np.transpose(a, (1, 0, 2))
    if o['flip_i']:
        a = a[::-1]
    if o['flip_j']:
        a = a[:, ::-1]
    if o['flip_k']:
        a = a[:, :, ::-1]
    return np.ascontiguousarray(a)

patients_table = pd.read_csv(INPUTS['patient_data'][0], dtype={'patient_id': str})
p_rows = patients_table[patients_table['phase'] == 'P'].drop_duplicates('patient_id')
def internal_descriptor(row):
    try:
        return orientation_descriptor(load_nii(DATA_DIR / row['ct_path']), load_nii(DATA_DIR / row['liver_mask_path']) > 0)
    except Exception as error:
        return {'error': repr(error)[:120]}
with ThreadPoolExecutor(8 if IN_COLAB else 2) as pool:
    internal_desc = pd.DataFrame([d or {} for d in pool.map(internal_descriptor, [r for _, r in p_rows.iterrows()])])
ok_int = internal_desc.dropna(subset=['liver_i', 'bone_i', 'bone_j'])
REF = {'front_back_axis_is_j': float((abs(ok_int['bone_j'] - .5) > abs(ok_int['bone_i'] - .5)).mean()),
       'liver_at_low_i': float((ok_int['liver_i'] < .5).mean()), 'bone_at_high_j': float((ok_int['bone_j'] > .5).mean()),
       'liver_bulk_at_low_k': float((ok_int['bulk_k'] < .5).mean()), 'n': int(len(ok_int))}
print('Internal orientation reference:', REF)
if not (REF['front_back_axis_is_j'] >= .8 and REF['liver_at_low_i'] >= .8 and REF['bone_at_high_j'] >= .8 and REF['liver_bulk_at_low_k'] >= .7):
    raise RuntimeError('STOP — the internal orientation reference differs from the 17 Oct check. Send this output.')
# Expected orientation of these files (Week 10b wrote them with SimpleITK from DICOM: LPS, slices feet -> head); listed for the summary
EXPECTED = {'transpose': False, 'flip_i': False, 'flip_j': False, 'flip_k': True}

# ---- per patient: E5 outputs (gates) and the axis-fixed variants from the same Week-10g arrays ----
def load_resampled(row, target_for):
    raw, spacing = {}, None
    for phase in PHASE_ORDER:
        ct, sp = read_nifti(row.get(f'ct_{phase}'))
        if ct is not None and spacing is None:
            spacing = sp
        raw[phase] = (ct, read_nifti(row.get(f'mask_{phase}'))[0], read_nifti(row.get(f'liver_mask_{phase}'))[0])
    spacing = spacing or (1.0, 1.0, 1.0)
    target = target_for(spacing)
    ct_r, tm_r, lm_r = {}, {}, {}
    for phase, (ct, tm, lm) in raw.items():
        ct_r[phase] = None if ct is None else resample_to_isotropic(ct, spacing, target)
        tm_r[phase] = None if (ct is None or tm is None) else resample_to_isotropic((tm > 0.5).astype(np.uint8), spacing, target, is_mask=True)
        lm_r[phase] = None if (ct is None or lm is None) else resample_to_isotropic((lm > 0.5).astype(np.uint8), spacing, target, is_mask=True)
    masks_present = [m for m in tm_r.values() if m is not None]
    if not masks_present:
        return None
    ref_shape = masks_present[0].shape
    tumor_union = np.zeros(ref_shape, dtype=np.uint8)
    for m in masks_present:
        if m.shape == ref_shape:
            tumor_union = np.maximum(tumor_union, m)
    liver, liver_fallback = None, False
    for phase in ['C2', 'C1', 'P', 'C3']:
        lm = lm_r.get(phase)
        if lm is not None and lm.shape == ref_shape and lm.sum() > 0:
            liver = lm
            break
    if liver is None:
        liver, liver_fallback = binary_dilation(tumor_union > 0, iterations=TUMOR_DILATION_MM).astype(np.uint8), True
    return {'ct': ct_r, 'tm': tm_r, 'ref_shape': ref_shape, 'tumor_union': tumor_union, 'liver': liver, 'liver_fallback': liver_fallback}

def e5_outputs(r):                                              # E5 cell 4 loop, unchanged
    ct_r, tm_r, ref_shape, tumor_union, liver = r['ct'], r['tm'], r['ref_shape'], r['tumor_union'], r['liver']
    out = {'tumor_voxels': int(tumor_union.sum()), 'available': [], 'tumor': np.zeros((4, *CROP_SIZE), np.uint8)}
    for window_name, window_fn in WINDOWS.items():
        cnn = np.zeros((4, *CROP_SIZE), dtype=np.float32)
        rad = np.zeros((4, *CROP_SIZE), dtype=np.float32)
        available = []
        for i, phase in enumerate(PHASE_ORDER):
            if ct_r[phase] is None or ct_r[phase].shape != ref_shape:
                continue
            crop, _ = crop_96_around_tumor(window_fn(ct_r[phase]) * (liver > 0), tumor_union)
            rad[i] = crop
            cnn[i] = zscore_nonzero(crop)
            available.append(phase)
            if tm_r[phase] is not None:
                out['tumor'][i] = crop_96_around_tumor(tm_r[phase], tumor_union)[0]
        out[window_name] = {'cnn': cnn} if window_name == 'historical_W150C50' else {'cnn': cnn, 'rad': rad}
        out['available'] = available
    return out

def descriptor_of(r):
    for phase in ['P', 'C2', 'C1', 'C3']:                         # non-contrast first (bone not confused with contrast)
        if r['ct'][phase] is not None and r['ct'][phase].shape == r['ref_shape']:
            to_ijk = lambda a: np.transpose(a, (2, 1, 0))
            d = orientation_descriptor(window_corrected(to_ijk(r['ct'][phase])), to_ijk(r['liver']) > 0)
            if d is not None:
                d['phase_used'] = phase
            return d
    return None

def axis_fixed(r, o, rules):
    """(k, j, i) Week-10g arrays -> (i, j, k), matched to the internal orientation; Week-10g or Week-2 crop rules."""
    to = lambda a: apply_orientation(np.transpose(a, (2, 1, 0)), o)
    tu, lv = to(r['tumor_union']), to(r['liver'])
    out = {'available': [], 'tumor': np.zeros((4, *CROP_SIZE), np.uint8), 'cnn': np.zeros((4, *CROP_SIZE), np.float32),
           'rad': np.zeros((4, *CROP_SIZE), np.float32 if rules == 'iso1mm' else np.uint8)}
    center = compute_center_of_mass(tu) or compute_center_of_mass(lv) or tuple(s // 2 for s in tu.shape)
    for i, phase in enumerate(PHASE_ORDER):
        ct = r['ct'][phase]
        if ct is None or ct.shape != r['ref_shape']:
            continue
        if rules == 'iso1mm':                                   # E5 rules, only the axes changed
            crop, _ = crop_96_around_tumor(window_corrected(to(ct)) * (lv > 0), tu)
            if r['tm'][phase] is not None:
                out['tumor'][i] = crop_96_around_tumor(to(r['tm'][phase]), tu)[0]
        else:                                                   # E6 primary rules: Week-2 crop, 0-255 uint8
            crop = np.clip(crop_cube(window_corrected(to(ct)) * (lv > 0), center, CROP_SIZE[0]), 0, 255).astype(np.uint8)
            if r['tm'][phase] is not None:
                out['tumor'][i] = (crop_cube(to(r['tm'][phase]), center, CROP_SIZE[0]) > 0).astype(np.uint8)
        out['rad'][i] = crop
        out['cnn'][i] = zscore_nonzero(crop.astype(np.float32))
        out['available'].append(phase)
    out['tumor_in_crop'] = float(out['tumor'].max(axis=0).sum() / max(1, tu.sum()))
    return out

def header_orientation(direction):
    """Amendment 18.1: map index axes to the internal convention (axis 0 -> patient's left, axis 1 -> posterior,
    axis 2 -> feet) from the LPS direction cosines in the image header. None if not an axial acquisition."""
    D = np.array(direction, dtype=float).reshape(3, 3)          # column c = physical direction of index axis c
    p = [int(np.argmax(np.abs(D[:, c]))) for c in range(3)]
    s = [float(np.sign(D[p[c], c])) for c in range(3)]
    if p[2] != 2 or sorted(p[:2]) != [0, 1]:
        return None
    transpose = p[0] == 1
    a0, a1 = (1, 0) if transpose else (0, 1)
    return {'transpose': bool(transpose), 'flip_i': bool(s[a0] < 0), 'flip_j': bool(s[a1] < 0), 'flip_k': bool(s[2] > 0)}

def reference_ct_path(row):                                     # the CT whose spacing Week 10g uses (first available)
    for phase in PHASE_ORDER:
        path = row.get(f'ct_{phase}')
        if isinstance(path, str) and path.strip() and path.strip().lower() not in {'nan', 'none', 'null'}:
            return path
    return None

def header_direction(path):
    reader = sitk.ImageFileReader()
    reader.SetFileName(path)
    reader.ReadImageInformation()
    return reader.GetDirection()

def preprocess(row):
    pid = str(row['patient_id'])
    iso = load_resampled(row, lambda spacing: TARGET_SPACING_MM)
    if iso is None:
        return pid, None
    result = e5_outputs(iso)
    direction = header_direction(reference_ct_path(row))
    o = header_orientation(direction)
    d = descriptor_of(iso)
    anatomy = decide(d) if d is not None and np.isfinite(d['bone_i']) else {}
    result['orientation'] = {'header_axial': o is not None, 'direction': [round(float(v), 3) for v in direction],
                             **(o or {}), **{f'anatomy_{k}': v for k, v in anatomy.items()},
                             **(d or {}), 'liver_fallback': iso['liver_fallback']}
    if o is None:
        return pid, result                                       # stopped below, after all patients are listed
    result['axis_fixed_iso1mm'] = axis_fixed(iso, o, 'iso1mm')
    del iso
    e6 = load_resampled(row, lambda spacing: (INPLANE_MM, INPLANE_MM, float(spacing[2])))
    result['axis_fixed_E6rules'] = axis_fixed(e6, o, 'E6rules')
    return pid, result

PRE = {}
t0 = time.time()
rows = [row for _, row in manifest.iterrows()]
with ThreadPoolExecutor(N_IO) as pool:
    for n, (pid, result) in enumerate(pool.map(preprocess, rows), start=1):
        if result is None:
            raise RuntimeError(f'STOP — {pid} could not be preprocessed.')
        PRE[pid] = result
        if n % 10 == 0:
            print(f'  preprocessed {n}/{len(rows)} ({(time.time() - t0) / 60:.1f} min)', flush=True)

orient = pd.DataFrame({pid: PRE[pid]['orientation'] for pid in ALL_IDS}).T
orient.to_csv(OUT_DIR / 'E5c_orientation_decisions.csv')
NOT_AXIAL = orient.index[~orient['header_axial'].astype(bool)].tolist()
if NOT_AXIAL:
    raise RuntimeError(f'STOP — image header is not axial for {NOT_AXIAL[:10]}. Send this output.')
ORIENTATION_SUMMARY = {'source': 'image header (amendment 18.1)', 'liver_mask_fallback': int(orient['liver_fallback'].astype(bool).sum())}
for part in EXPECTED:
    applied = orient[part].astype(bool)
    clear = orient[f'anatomy_conf_{part}'].fillna(False).astype(bool) if f'anatomy_conf_{part}' in orient else pd.Series(False, index=orient.index)
    agree = (orient.loc[clear, f'anatomy_{part}'].astype(bool) == applied[clear]) if clear.any() else pd.Series(dtype=bool)
    ORIENTATION_SUMMARY[part] = {'applied_to': int(applied.sum()), 'anatomy_clear': int(clear.sum()),
                                 'anatomy_agrees_with_header': int(agree.sum())}
ORIENTATION_SUMMARY['directions'] = orient['direction'].astype(str).value_counts().head(5).to_dict()
print('Orientation decisions:', json.dumps(ORIENTATION_SUMMARY, indent=1))
VARIANTS = ['historical_W150C50', 'corrected_-200_200', 'axis_fixed_iso1mm', 'axis_fixed_E6rules']
print('Tumour fraction inside the crop (median):', {v: round(float(np.median([PRE[p][v]['tumor_in_crop'] for p in ALL_IDS])), 3)
                                                    for v in VARIANTS[2:]})

In [ ]:
# Cell 5 — CNN inference on every variant; gate 1 (submitted results) and gate 2 (E5 v2 results)
def make_cnn(in_channels):
    return resnet18(spatial_dims=3, n_input_channels=in_channels, num_classes=3, feed_forward=True)

def load_models(keys, in_channels):
    models = []
    for key in keys:
        checkpoint = torch.load(MODEL_FILES[key], map_location=DEVICE, weights_only=False)
        model = make_cnn(in_channels).to(DEVICE)
        model.load_state_dict(checkpoint['state_dict'])
        model.eval()
        models.append(model)
    return models

CNNS = {'single_phase_CNN': (load_models([f'w3_{f}' for f in range(5)], 1), [2]),
        'original_CNN': (load_models([f'w4_{f}' for f in range(5)], 4), [0, 1, 2, 3]),
        'phase_dropout_CNN': (load_models([f'd_{f}' for f in range(5)], 4), [0, 1, 2, 3])}

def part(pid, variant, key):
    if variant in WINDOWS:
        return PRE[pid][variant][key] if key in ('cnn', 'rad') else PRE[pid][key]
    return PRE[pid][variant][key]

@torch.no_grad()
def cnn_probs(models, channels, variant):
    rows = {}
    for pid in ALL_IDS:
        x = torch.from_numpy(part(pid, variant, 'cnn')[channels]).unsqueeze(0).to(DEVICE)
        rows[pid] = np.mean([torch.softmax(m(x), dim=1).float().cpu().numpy()[0] for m in models], axis=0)
    return pd.DataFrame.from_dict(rows, orient='index', columns=['p_HCC', 'p_ICC', 'p_CHCC'])

PROBS = {}
for name, (models, channels) in CNNS.items():
    for variant in VARIANTS:
        PROBS[(name, variant)] = cnn_probs(models, channels, variant)

hist = pd.read_csv(INPUTS['historical_external'][0]).set_index('patient_id')
gate = {}
for name, prefix in [('original_CNN', 'w4'), ('single_phase_CNN', 'w3')]:
    ours = PROBS[(name, 'historical_W150C50')].loc[hist.index].to_numpy()
    saved = hist[[f'{prefix}_p_HCC', f'{prefix}_p_ICC', f'{prefix}_p_CHCC']].to_numpy()
    gate[name] = {'max_prob_diff': float(np.abs(ours - saved).max()),
                  'hcc_count_ours': int((ours.argmax(1) == 0).sum()), 'hcc_count_saved': int((saved.argmax(1) == 0).sum())}
REPRODUCED = all(g['max_prob_diff'] < 0.02 and g['hcc_count_ours'] == g['hcc_count_saved'] for g in gate.values())

E5_SAVED = pd.read_csv(INPUTS['e5_v2_predictions'][0])
def e5_saved(model, window):
    t = E5_SAVED[(E5_SAVED['model'] == model) & (E5_SAVED['window'] == window)].set_index('patient_id')
    return t.loc[ALL_IDS, ['p_HCC', 'p_ICC', 'p_CHCC']].to_numpy()
gate2 = {f'{name} / {w}': float(np.abs(PROBS[(name, w)].loc[ALL_IDS].to_numpy() - e5_saved(name, w)).max())
         for name in CNNS for w in WINDOWS}
print('Gate 1 (submitted):', json.dumps(gate, indent=1))
print('Gate 2 (E5 v2), max probability differences:', json.dumps(gate2, indent=1))
GATE2_OK = all(v < 0.02 for v in gate2.values())
if IN_COLAB and not (REPRODUCED and GATE2_OK):
    raise RuntimeError('STOP — a reproduction gate failed. Send this output for review.')

In [ ]:
# Cell 6 — Radiomics without shape features (E2 models) on the E5 and the axis-fixed variants
def make_extractor():
    extractor = featureextractor.RadiomicsFeatureExtractor(binWidth=5)
    extractor.disableAllImageTypes()
    extractor.enableImageTypeByName('Original')
    extractor.disableAllFeatures()
    for feature_class in ('shape', 'firstorder', 'glcm', 'glrlm', 'glszm', 'gldm', 'ngtdm'):
        extractor.enableFeatureClassByName(feature_class)
    return extractor

def extract_arrays(pid, rad4, tumor4, available):
    extractor = make_extractor()
    features = {}
    for i, phase in enumerate(PHASE_ORDER):
        if phase not in available or tumor4[i].sum() == 0:
            continue
        ct_img = sitk.GetImageFromArray(rad4[i].astype(np.float32)); ct_img.SetSpacing((1.0, 1.0, 1.0))
        mask_img = sitk.GetImageFromArray(tumor4[i].astype(np.uint8)); mask_img.SetSpacing((1.0, 1.0, 1.0))
        try:
            for key, value in extractor.execute(ct_img, mask_img).items():
                if not key.startswith('diagnostics'):
                    try:
                        features[f'{phase}_{key}'] = float(value)
                    except (TypeError, ValueError):
                        pass
        except Exception as error:
            print(f'  radiomics failed for {pid}/{phase}: {error}')
    return pid, features

RAD_VARIANTS = ['corrected_-200_200', 'axis_fixed_iso1mm', 'axis_fixed_E6rules']
RAD = {}
t0 = time.time()
for variant in RAD_VARIANTS:
    jobs = [(pid, part(pid, variant, 'rad'), part(pid, variant, 'tumor'), part(pid, variant, 'available')) for pid in ALL_IDS]
    results = None
    if N_JOBS > 1:
        try:
            results = joblib.Parallel(n_jobs=N_JOBS)(joblib.delayed(extract_arrays)(*job) for job in jobs)
        except Exception as error:
            print(f'  parallel radiomics failed ({error!r:.120}); continuing one patient at a time')
    if results is None:
        results = [extract_arrays(*job) for job in jobs]
    RAD[variant] = pd.DataFrame.from_dict(dict(results), orient='index').reindex(ALL_IDS)
    RAD[variant].to_csv(OUT_DIR / f'E5c_radiomics_features_{variant}.csv')
    print(f'  radiomics done for {variant}: {RAD[variant].shape} ({(time.time() - t0) / 60:.1f} min)', flush=True)

internal = pd.read_parquet(INPUTS['radiomics_features'][0])
internal['patient_id'] = internal['patient_id'].astype(str)
split = json.loads(INPUTS['split'][0].read_text())
dev_ids = [str(x) for x in split['dev_ids']]
fold_splits = [json.loads(INPUTS[f'fold_{f}'][0].read_text()) for f in range(5)]
rad_models = [joblib.load(MODEL_FILES[f'rad_{f}']) for f in range(5)]

def radiomics_probs(features):                                   # E5 'real_plus_median', unchanged in effect
    per_fold, imputed, total = [], 0, 0
    for f, m in enumerate(rad_models):
        names = m['sel_names']
        train_ids = [p for p in dev_ids if p not in set(fold_splits[f]['val_ids'])]
        medians = internal.set_index('patient_id').loc[train_ids, names].median(axis=0, skipna=True)
        X = features.reindex(columns=names).loc[ALL_IDS].to_numpy(dtype=np.float64)
        missing = ~np.isfinite(X)
        imputed += int(missing.sum()); total += X.size
        X[missing] = np.broadcast_to(medians.to_numpy(), X.shape)[missing]
        per_fold.append(m['clf'].predict_proba(m['scaler'].transform(m['imputer'].transform(X.astype(np.float32)))))
    return pd.DataFrame(np.mean(per_fold, axis=0), index=ALL_IDS, columns=['p_HCC', 'p_ICC', 'p_CHCC']), imputed / total

IMPUTED = {}
for variant in RAD_VARIANTS:
    PROBS[('radiomics_noshape', variant)], IMPUTED[variant] = radiomics_probs(RAD[variant])
gate2['radiomics_noshape / corrected_-200_200'] = float(np.abs(PROBS[('radiomics_noshape', 'corrected_-200_200')].loc[ALL_IDS].to_numpy()
                                                              - e5_saved('radiomics_noshape', 'corrected_-200_200')).max())
print('Radiomics gate 2 difference:', gate2['radiomics_noshape / corrected_-200_200'], '| imputed fraction:', IMPUTED)
if IN_COLAB and gate2['radiomics_noshape / corrected_-200_200'] > 1e-4:
    raise RuntimeError('STOP — radiomics did not reproduce E5 v2. Send this output.')

In [ ]:
# Cell 7 — Fusion exactly as in E5, on the E5 and the axis-fixed variants
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
W4 = ['w4_p_HCC', 'w4_p_ICC', 'w4_p_CHCC']
W5 = ['w5_p_HCC', 'w5_p_ICC', 'w5_p_CHCC']
FUSIONS = {'cnn_plus_radiomics': W4 + W5, 'full_fusion': W4 + W5 + ['age', 'gender']}

oof_original = pd.read_csv(INPUTS['e2_oof_meta'][0], dtype={'patient_id': str})
d_pred = pd.read_csv(INPUTS['e3_d_predictions'][0], dtype={'patient_id': str})
d_oof = d_pred.query("pattern == 'complete' and split == 'oof'").set_index('patient_id')[['p_HCC', 'p_ICC', 'p_CHCC']]
oof_dropout = oof_original.copy()
oof_dropout[W4] = d_oof.loc[oof_dropout['patient_id']].to_numpy()
OOF_BY_CNN = {'original_CNN': oof_original, 'phase_dropout_CNN': oof_dropout}

def prepare(X):
    X = X.astype(np.float32)
    X[X <= -1] = np.nan
    return X

def fit_meta_folds(columns, oof):
    fitted = []
    for f in range(5):
        train = oof['fold'].astype(int).ne(f)
        X = prepare(oof.loc[train, columns].to_numpy())
        imputer = SimpleImputer(strategy='median').fit(X)
        scaler = StandardScaler().fit(imputer.transform(X))
        model = LogisticRegression(max_iter=2000, C=1.0, class_weight='balanced', random_state=SEED)
        fitted.append((imputer, scaler, model.fit(scaler.transform(imputer.transform(X)), oof.loc[train, 'label'].astype(int))))
    return fitted

CLINICAL_SCENARIOS = {
    'observed_TCIA': (clinical_ext.set_index('patient_id').loc[ALL_IDS, 'age'].to_numpy(),
                      clinical_ext.set_index('patient_id').loc[ALL_IDS, 'sex'].to_numpy()),
    'both_missing_fold_median': (np.full(len(ALL_IDS), np.nan), np.full(len(ALL_IDS), np.nan)),
}
FITTED = {(cnn, fusion): fit_meta_folds(cols, oof) for cnn, oof in OOF_BY_CNN.items() for fusion, cols in FUSIONS.items()}
for variant in RAD_VARIANTS:
    for (cnn_name, fusion_name), fitted in FITTED.items():
        base = pd.DataFrame(index=ALL_IDS)
        base[W4] = PROBS[(cnn_name, variant)].loc[ALL_IDS].to_numpy()
        base[W5] = PROBS[('radiomics_noshape', variant)].loc[ALL_IDS].to_numpy()
        scenarios = CLINICAL_SCENARIOS if fusion_name == 'full_fusion' else {'none': None}
        for scenario, values in scenarios.items():
            frame = base.copy()
            if values is not None:
                frame['age'], frame['gender'] = values
            X = prepare(frame[FUSIONS[fusion_name]].to_numpy())
            probs = np.mean([model.predict_proba(scaler.transform(imputer.transform(X))) for imputer, scaler, model in fitted], axis=0)
            label = f'{fusion_name} [{cnn_name}]' + ('' if scenario == 'none' else f' {{{scenario}}}')
            PROBS[(label, variant)] = pd.DataFrame(probs, index=ALL_IDS, columns=['p_HCC', 'p_ICC', 'p_CHCC'])
for label in ['full_fusion [phase_dropout_CNN] {observed_TCIA}', 'cnn_plus_radiomics [phase_dropout_CNN]']:
    gate2[f'{label} / corrected_-200_200'] = float(np.abs(PROBS[(label, 'corrected_-200_200')].loc[ALL_IDS].to_numpy()
                                                          - e5_saved(label, 'corrected_-200_200')).max())
print('Fusion gate 2 differences:', {k: v for k, v in gate2.items() if 'fusion' in k or 'plus' in k})

In [ ]:
# Cell 8 — Results (exact intervals), saved outputs and the block to send back
def exact_ci(k, n):
    low = 0.0 if k == 0 else float(beta.ppf(0.025, k, n - k + 1))
    high = 1.0 if k == n else float(beta.ppf(0.975, k + 1, n - k))
    return low, high

SUBSETS = {'clean_89': CLEAN_IDS, 'all_103': ALL_IDS}
rows = []
for (model, variant), probs in PROBS.items():
    for subset, ids in SUBSETS.items():
        p = probs.loc[ids].to_numpy()
        pred = p.argmax(axis=1)
        k, n = int((pred == 0).sum()), len(ids)
        low, high = exact_ci(k, n)
        rows.append({'model': model, 'variant': variant, 'subset': subset, 'n': n, 'hcc_correct': k,
                     'hcc_sensitivity': k / n, 'ci_low': low, 'ci_high': high,
                     'pred_HCC': k, 'pred_ICC': int((pred == 1).sum()), 'pred_CHCC': int((pred == 2).sum()),
                     'mean_p_HCC': float(p[:, 0].mean())})
results = pd.DataFrame(rows)
results.to_csv(OUT_DIR / 'E5c_results.csv', index=False)

long = []
for (model, variant), probs in PROBS.items():
    t = probs.copy()
    t.insert(0, 'patient_id', t.index)
    t.insert(0, 'variant', variant)
    t.insert(0, 'model', model)
    t['mixed_scan_patient'] = t['patient_id'].isin(MIXED_IDS)
    long.append(t)
pd.concat(long, ignore_index=True).to_csv(OUT_DIR / 'E5c_predictions.csv', index=False)

summary = {'run_id': RUN_ID, 'notebook': 'E5c_external_HCC_TACE_Seg_axis_corrected_v2.ipynb', 'versions': VERSIONS,
           'input_sha256': input_hashes, 'gate1_submitted': gate, 'gate1_reproduced': REPRODUCED, 'gate2_e5_v2': gate2,
           'orientation': ORIENTATION_SUMMARY, 'internal_orientation_reference': REF, 'radiomics_imputed_fraction': IMPUTED,
           'mixed_scan_patients': sorted(MIXED_IDS), 'clean_n': len(CLEAN_IDS),
           'protocol': '00 log section 18 (approved 17 Oct 2026); primary corrected = axis_fixed_E6rules, clean_89; argmax'}
(OUT_DIR / 'summary.json').write_text(json.dumps(summary, indent=2, default=str))
manifest_out = [{'file': p.relative_to(OUT_DIR).as_posix(), 'bytes': p.stat().st_size, 'sha256': sha256_of(p)}
                for p in sorted(OUT_DIR.rglob('*')) if p.is_file()]
pd.DataFrame(manifest_out).to_csv(OUT_DIR / 'output_manifest.csv', index=False)

MODEL_ORDER = ['phase_dropout_CNN', 'original_CNN', 'single_phase_CNN', 'radiomics_noshape',
               'cnn_plus_radiomics [phase_dropout_CNN]', 'full_fusion [phase_dropout_CNN] {observed_TCIA}',
               'full_fusion [phase_dropout_CNN] {both_missing_fold_median}', 'cnn_plus_radiomics [original_CNN]',
               'full_fusion [original_CNN] {observed_TCIA}', 'full_fusion [original_CNN] {both_missing_fold_median}']
print('\n================ COPY FROM HERE AND SEND BACK ================')
print(f'Run E5c_{RUN_ID} | gate 1 (submitted 0/103, 5/103) reproduced: {REPRODUCED} | gate 2 (E5 v2) max diff: '
      f'{max(gate2.values()):.1e} | clean n: {len(CLEAN_IDS)}')
print('Orientation:', ORIENTATION_SUMMARY)
for subset in ['clean_89', 'all_103']:
    t = results[results['subset'] == subset].copy()
    t['cell'] = t.apply(lambda r: f"{int(r['hcc_correct'])}/{int(r['n'])} ({r['hcc_sensitivity']:.2f})", axis=1)
    table = t.pivot(index='model', columns='variant', values='cell').reindex(MODEL_ORDER)[VARIANTS]
    print(f'\n--- HCC recognised, {subset} (E5 = corrected_-200_200; primary corrected = axis_fixed_E6rules) ---')
    print(table.fillna('-').to_string())
prim = results[(results['variant'] == 'axis_fixed_E6rules') & (results['subset'] == 'clean_89')].set_index('model')
print('\nPrimary corrected, clean 89, exact 95% CI:')
for model in MODEL_ORDER:
    r = prim.loc[model]
    print(f"  {model:58s} {int(r['hcc_correct'])}/89  {r['hcc_sensitivity']:.3f} ({r['ci_low']:.3f}-{r['ci_high']:.3f})  "
          f"pred H/I/C {int(r['pred_HCC'])}/{int(r['pred_ICC'])}/{int(r['pred_CHCC'])}")
print('\nVersions:', VERSIONS)
print('==============================================================')